# Ejercicio: Extracción de datos de la API de TMDB (The Movie Database)

![imagen](https://www.themoviedb.org/assets/2/v4/logos/v2/blue_square_2-d537fb228cf3ded904ef09b136fe3fec72548ebc1fea3fbbd1ad9e36364db38b.svg)

Trabajaremos con **TMDB**, una de las bases de datos de cine más importantes del mundo. El objetivo es extraer información de películas para realizar un análisis de mercado inicial.

### 1. Registro y Documentación
Tendrás que [registrarte en TMDB](https://www.themoviedb.org/signup) para obtener tu **API Key (v3 auth)** y consultar la [documentación oficial](https://developer.themoviedb.org/reference/intro/getting-started).

### 2. Objetivo del Ejercicio
Queremos que consultes la API para que te devuelva la información de las películas que empiecen por la **inicial de tu nombre** (parámetro `query`). 

Debes guardar la información en un archivo `.csv` con la siguiente estructura de columnas:

| Columna | Descripción |
| :--- | :--- |
| **id** | ID interno de la película en TMDB |
| **title** | Título de la película |
| **release_date** | Fecha de estreno |
| **genres** | Nombres de los géneros (ej: "Acción, Comedia") |
| **vote_average** | Puntuación media de los usuarios |
| **overview** | Sinopsis o resumen de la trama |

### 3. El Reto: Mapeo de Géneros
A diferencia de otras APIs, el endpoint de búsqueda de películas devuelve los géneros como una lista de IDs numéricos (ej: `[28, 12]`). 

**Tu labor es:**
1. Consultar el endpoint de "Genre List" para obtener la relación entre IDs y nombres.
2. Sustituir los IDs en tu DataFrame final por los nombres reales de los géneros (separados por comas).

---

### Código de inicio
Aquí tienes el bloque para empezar a configurar tus llamadas:

```python
import requests
import pandas as pd

# Rellena estas variables
api_key = "TU_API_KEY_AQUÍ"
url_base = "[https://api.themoviedb.org/3](https://api.themoviedb.org/3)"
mi_inicial = "P" # Sustituye por tu inicial

In [1]:
import requests
import pandas as pd

# Configuración
api_key = "9d7ecdec6d98ae084568cbf4cf95c3f8"
url_base = "https://api.themoviedb.org/3"
mi_inicial = "I" # Cambiar por la inicial del alumno

# 1. Obtener el diccionario de géneros (ID -> Nombre)
# Tip: Endpoint /genre/movie/list
def get_genres_map(api_key):
    # Tu código aquí
    pass

# 2. Buscar películas
# Tip: Endpoint /search/movie
def search_movies(api_key, query):
    # Tu código aquí
    # Recordad gestionar la paginación si queréis más de 20 resultados
    pass

# 3. Procesamiento y Limpieza
# - Mapear los IDs de géneros a nombres reales.
# - Crear el DataFrame.
# - Exportar a CSV.

In [2]:
# idioma de los resultados en español de España
idioma = "es-ES"

In [3]:
# función de obtener el el mapeo de géneros:

def get_genres_map(api_key, language="es-ES"):
    """ obtiene el diccionario de géneros de películas desde la API de TMDB.  
    parámetros: api_key (calve), language (idioma de los géneros)
    me devuelve: un diccionario {id_genero: nombre_genero}"""
    endpoint = "/genre/movie/list" # endpoint para obtener la lista de géneros
    params = {"api_key": api_key, "language": language} # parámetros de la petición
    
    respuesta = requests.get(url_base + endpoint, params=params) # petición a la API
    if respuesta.status_code == 200: # si la petición es existosa, entonces:
        datos = respuesta.json()
        generos_map = {genero["id"]: genero["name"] for genero in datos["genres"]} # creamos el diccionario: {id: nombre}; usamos un diccionario por comprensión
        return generos_map
    else:
        print(f"Error con código: {respuesta.status_code}, {respuesta.text}")
        return


# ejecuto la función:

diccionario_generos = get_genres_map(api_key, idioma) # llamo a la función y guardo el resultado

print("Algunos géneros disponibles:") # muestro algunos géneros como ejemplo
for i, (id_gen, nombre_gen) in enumerate(diccionario_generos.items()):
    if i < 10:  # sólo los 10 primeros
        print(f"ID {id_gen}: {nombre_gen}")

Algunos géneros disponibles:
ID 28: Acción
ID 12: Aventura
ID 16: Animación
ID 35: Comedia
ID 80: Crimen
ID 99: Documental
ID 18: Drama
ID 10751: Familia
ID 14: Fantasía
ID 36: Historia


In [4]:
# función de buscar películas

def search_movies(api_key, query, language="es-ES", max_pages=5):
    """busco películas por título en la API de TMDB  
    parámetros: api_key, query (término de búsqueda como la inicial del nombre), language, max_pages (num máx de págs a obtener (x defecto: 5)
    devuelve: lista de diccionarios con la información de las películas
    """
    endpoint = "/search/movie"
    todas_peliculas = [] # lista pa guardar las pelis
    
    for pagina in range(1, max_pages + 1): # hago un bucle para obtener múltiples páginas de resultados
        params = {"api_key": api_key, "query": query,"language": language,"page": pagina}
        respuesta = requests.get(url_base + endpoint, params=params)
        if respuesta.status_code == 200:
            datos = respuesta.json()
        
            resultados_pagina = datos.get("results", []) # obtengo los tesultados
        
            if not resultados_pagina: # si no hay más resultados, salimos del bucle
                break
            
            todas_peliculas.extend(resultados_pagina) # añado las películas a la lista
            
            print(f"Página {pagina}: {len(resultados_pagina)} películas encontradas.")
            
            if pagina >= datos.get("total_pages", 1): # si llego a la última página, salgo del bucle
                break
        else:
            print(f"Error en la página {pagina}. Código: {respuesta.status_code}")
            break
    
    print(f"Películas obtenidas: {len(todas_peliculas)}")
    return todas_peliculas



In [5]:
# ejecuto la búsqueda

# busco películas que empiecen por mi inicial
# max_pages=3 obtendrá hasta 60 películas (20 por página)
lista_peliculas = search_movies(api_key, mi_inicial, idioma, max_pages=3)

if lista_peliculas: # muestro las 3 primeras películas como ejemplo
    for i, pelicula in enumerate(lista_peliculas[:3]):
        print(f"{i+1}. {pelicula.get('title', 'N/A')}")
        print(f"Fecha: {pelicula.get('release_date', 'N/A')}")
        print(f"Géneros: {pelicula.get('genre_ids', [])}")
        print(f"Puntuación: {pelicula.get('vote_average', 'N/A')}")

Página 1: 20 películas encontradas.
Página 2: 20 películas encontradas.
Página 3: 20 películas encontradas.
Películas obtenidas: 60
1. Peaky Blinders: El hombre inmortal
Fecha: 2026-03-05
Géneros: [80, 18]
Puntuación: 7.431
2. Guardianes de la noche: Kimetsu no Yaiba La fortaleza infinita
Fecha: 2025-07-18
Géneros: [16, 28, 14]
Puntuación: 7.7
3. Vengadores: Infinity War
Fecha: 2018-04-25
Géneros: [12, 28, 878]
Puntuación: 8.234


In [6]:
# función de convertir los ids de géneros a nombres

def convertir_generos(ids_generos, diccionario_generos):
    """convierto una lista de ids de géneros a un str con los nombres.
    parámetros: ids_generos, diccionario_generos ({id: nombre})    
    devuelve: cadena de texto con los nombres separados por comas (ej: "Acción, Aventura, Fantasía")
    """
    
    if not ids_generos: # si la lista está vacía o es None, devuelve "N/A"
        return "N/A"
    
    # convierto cada id a su nombre correspondiente:
    nombres_generos = [diccionario_generos.get(id_gen, f"Desconocido ({id_gen})") # uso .get() para manejar ids que no estén en el diccionario
                       for id_gen in ids_generos]
    
    return ", ".join(nombres_generos) # uno los nombres con ", " y devuelvo el resultado



In [7]:
# proceso las películas

peliculas_procesadas = [] # lista para guardar las películas procesadas

for pelicula in lista_peliculas:
    
    # Extraemos cada campo que nos interesa
    pelicula_procesada = {"id": pelicula.get("id", "N/A"),
        "title": pelicula.get("title", "N/A"),
        "release_date": pelicula.get("release_date", "N/A"),
        "genres": convertir_generos(pelicula.get("genre_ids", []), diccionario_generos),
        "vote_average": pelicula.get("vote_average", "N/A"),
        "overview": pelicula.get("overview", "N/A")
    }
    
    peliculas_procesadas.append(pelicula_procesada) # añado la película procesada a la lista

print(f"{len(peliculas_procesadas)} películas procesadas correctamente.")


if peliculas_procesadas: # muestro un ej emplo del resultado
    print(peliculas_procesadas[0])

60 películas procesadas correctamente.
{'id': 875828, 'title': 'Peaky Blinders: El hombre inmortal', 'release_date': '2026-03-05', 'genres': 'Crimen, Drama', 'vote_average': 7.431, 'overview': 'El gánster Tommy Shelby regresa a Birmingham para salvar a su familia —y a su país— cuando su hijo, del que se ha distanciado, se ve envuelto en un complot nazi.'}


In [9]:
# creo el dataframe a partir de la lista de pelis procesadas
# pandas convierte automáticamente la lista de dicc en una tabla

df_peliculas = pd.DataFrame(peliculas_procesadas)

df_peliculas

,id,title,release_date,genres,vote_average,overview
0,875828,Peaky Blinders: El hombre inmortal,2026-03-05,"Crimen, Drama",7.431,El gánster Tommy Shelby regresa a Birmingham p...
1,1311031,Guardianes de la noche: Kimetsu no Yaiba La fo...,2025-07-18,"Animación, Acción, Fantasía",7.700,El Cuerpo de Cazadores de Demonios se enfrenta...
2,299536,Vengadores: Infinity War,2018-04-25,"Aventura, Acción, Ciencia ficción",8.234,El todopoderoso Thanos ha despertado con la pr...
3,278154,Ice Age: El gran cataclismo,2016-06-23,"Aventura, Animación, Familia, Comedia, Ciencia...",6.100,En la nueva historia veremos a Scrat y su míti...
4,1424965,Infierno bajo cero,2026-02-24,"Drama, Suspense",5.200,
5,575265,Misión: Imposible - Sentencia final,2025-05-17,"Acción, Suspense, Aventura",7.200,El agente Ethan Hunt continúa su misión de imp...
6,1022789,Del revés 2 (Inside Out 2),2024-06-11,"Animación, Aventura, Comedia, Familia",7.500,Riley entra en la adolescencia y el Cuartel Ge...
7,240,El Padrino Parte II,1974-12-20,"Drama, Crimen",8.572,Continuación de la saga de los Corleone con do...
8,150540,Del revés (Inside Out),2015-06-17,"Animación, Familia, Aventura, Drama, Comedia",7.910,Riley es una chica que disfruta o padece toda ...
9,1032892,En un instante,2026-01-26,"Ciencia ficción, Drama",5.100,Tres historias interconectadas reflexionan sob...


In [9]:
df_peliculas.head()

,id,title,release_date,genres,vote_average,overview
0,875828,Peaky Blinders: El hombre inmortal,2026-03-05,"Crimen, Drama",7.460,El gánster Tommy Shelby regresa a Birmingham p...
1,1311031,Guardianes de la noche: Kimetsu no Yaiba La fo...,2025-07-18,"Animación, Acción, Fantasía",7.700,El Cuerpo de Cazadores de Demonios se enfrenta...
2,1205225,Las Puertas del Infierno,2025-04-08,"Terror, Suspense, Acción",5.892,Un grupo de amigos intenta escapar de la ciuda...
3,299536,Vengadores: Infinity War,2018-04-25,"Aventura, Acción, Ciencia ficción",8.234,El todopoderoso Thanos ha despertado con la pr...
4,1424965,Infierno bajo cero,2026-02-24,"Drama, Suspense",5.192,


In [10]:
df_peliculas.tail()

,id,title,release_date,genres,vote_average,overview
55,18874,Ghost In The Shell. Stand alone complex. Solid...,2007-09-01,"Animación, Acción, Suspense, Crimen, Ciencia f...",7.500,"Es el año 2034, han pasado dos años desde que ..."
56,177677,Misión imposible: Nación secreta,2015-07-28,"Acción, Aventura",7.221,Con la FMI disuelta y Ethan Hunt abandonado a ...
57,843527,La idea de tenerte,2024-05-02,"Romance, Drama, Comedia",7.313,"Solène, madre soltera de 40 años, comienza un ..."
58,11411,Superman IV: En busca de la paz,1987-07-24,"Acción, Aventura, Ciencia ficción",4.593,Poco despues de que la cumbre USA-URSS para el...
59,402,Instinto básico,1992-03-20,"Suspense, Misterio",6.934,Un antiguo cantante de rock ha sido brutalment...


In [11]:
df_peliculas.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            60 non-null     int64  
 1   title         60 non-null     str    
 2   release_date  60 non-null     str    
 3   genres        60 non-null     str    
 4   vote_average  60 non-null     float64
 5   overview      60 non-null     str    
dtypes: float64(1), int64(1), str(4)
memory usage: 2.9 KB


In [12]:
df_peliculas["vote_average"].describe() # estadísticas de las columnas numéricas

count    60.000000
mean      6.715417
std       1.660884
min       0.000000
25%       6.539000
50%       7.176500
75%       7.500000
max       8.572000
Name: vote_average, dtype: float64

In [13]:
# exportar a CSV

nombre_archivo = f"peliculas_inicial_{mi_inicial}.csv" # nombre del archivo de salida, uso la inicial para nombrar el archivo

# exporto el DataFrame a CSV
# index=False significa que no guardoel índice numérico del DataFrame
# encoding='utf-8-sig' asegura que los caracteres especiales (ñ, acentos) se guarden correctamente
df_peliculas.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')



In [14]:
# verifico el archivo creado

df_verificacion = pd.read_csv(nombre_archivo, encoding='utf-8-sig') # leo el archivo CSV para verificar que se creó correctamente

print(f"Número de filas: {len(df_verificacion)}")
print(f"Columnas: {list(df_verificacion.columns)}")


Número de filas: 60
Columnas: ['id', 'title', 'release_date', 'genres', 'vote_average', 'overview']


In [15]:
df_verificacion.head() # muestro las primeras filas del CSV

,id,title,release_date,genres,vote_average,overview
0,875828,Peaky Blinders: El hombre inmortal,2026-03-05,"Crimen, Drama",7.460,El gánster Tommy Shelby regresa a Birmingham p...
1,1311031,Guardianes de la noche: Kimetsu no Yaiba La fo...,2025-07-18,"Animación, Acción, Fantasía",7.700,El Cuerpo de Cazadores de Demonios se enfrenta...
2,1205225,Las Puertas del Infierno,2025-04-08,"Terror, Suspense, Acción",5.892,Un grupo de amigos intenta escapar de la ciuda...
3,299536,Vengadores: Infinity War,2018-04-25,"Aventura, Acción, Ciencia ficción",8.234,El todopoderoso Thanos ha despertado con la pr...
4,1424965,Infierno bajo cero,2026-02-24,"Drama, Suspense",5.192,NaN
